# Treinamento do Prophet para BTC/USDT

Este notebook lê `data/btc_usd_2024_2025.csv`, prepara `ds` (data) e `y` (fechamento), avalia o Prophet em uma divisão cronológica e salva o modelo em `artifacts/prophet_model.json`. O CSV atual foi baixado do CryptoDataDownload (Binance BTCUSDT) após o Yahoo Finance limitar as requisições. A previsão é experimental.

In [1]:
from pathlib import Path
import json
import pandas as pd
from prophet import Prophet
from prophet.serialize import model_to_json
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

DATA_PATH = Path('../data/btc_usd_2024_2025.csv')
SOURCE_PATH = Path('../data/data_source.json')
ARTIFACT_DIR = Path('../artifacts')
ARTIFACT_PATH = ARTIFACT_DIR / 'prophet_model.json'
METADATA_PATH = ARTIFACT_DIR / 'training_metadata.json'
data_source = json.loads(SOURCE_PATH.read_text(encoding='utf-8')) if SOURCE_PATH.exists() else {'source':'Fonte anotada manualmente; consulte README'}

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importing plotly failed. Interactive plots will not work.


## 1. Carregar e preparar os dados

In [2]:
raw = pd.read_csv(DATA_PATH, parse_dates=['Date'])
series = raw[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})
series = series.dropna().sort_values('ds').drop_duplicates(subset='ds').reset_index(drop=True)
assert len(series) >= 700, f'Esperava pelo menos 700 registros, encontrei {len(series)}.'
print(f'Registros: {len(series)}')
print(f'Período: {series.ds.min().date()} até {series.ds.max().date()}')
series.head()

Registros: 731
Período: 2024-01-01 até 2025-12-31


,ds,y
0,2024-01-01,44179.55
1,2024-01-02,44946.91
2,2024-01-03,42845.23
3,2024-01-04,44151.10
4,2024-01-05,44145.11


## 2. Divisão cronológica

Uso os primeiros 80% dos dias para treino e os 20% finais para teste. Não embaralho a série para que o modelo não veja datas futuras durante o treino.

In [3]:
split_index = int(len(series) * 0.8)
train = series.iloc[:split_index].copy()
test = series.iloc[split_index:].copy()
print(f'Treino: {len(train)} linhas, até {train.ds.max().date()}')
print(f'Teste: {len(test)} linhas, de {test.ds.min().date()} até {test.ds.max().date()}')

Treino: 584 linhas, até 2025-08-06
Teste: 147 linhas, de 2025-08-07 até 2025-12-31


## 3. Treino e avaliação

Faço uma avaliação walk-forward de um dia à frente: em cada dia de teste, ajusto o Prophet com os dados disponíveis até o dia anterior, prevejo o fechamento seguinte e só então acrescento o fechamento real ao histórico. A baseline recebe exatamente a mesma informação e usa o fechamento do dia anterior. MAE mede o erro médio em dólares e MAPE o erro percentual médio.

In [4]:
actual = test['y'].to_numpy()
walk_forward_predictions = []
history = train.copy()
for position in range(split_index, len(series)):
    current_day = series.iloc[[position]]
    one_day_model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
    one_day_model.fit(history)
    next_close = one_day_model.predict(current_day[['ds']])['yhat'].iloc[0]
    walk_forward_predictions.append(next_close)
    history = pd.concat([history, current_day], ignore_index=True)
predicted = walk_forward_predictions
baseline = series['y'].shift(1).iloc[split_index:].to_numpy()
metrics = pd.DataFrame([
    {'modelo':'Prophet', 'MAE_USD':mean_absolute_error(actual, predicted), 'MAPE_percent':mean_absolute_percentage_error(actual, predicted)*100},
    {'modelo':'Baseline (dia anterior)', 'MAE_USD':mean_absolute_error(actual, baseline), 'MAPE_percent':mean_absolute_percentage_error(actual, baseline)*100}
])
metrics.round(2)

18:18:52 - cmdstanpy - INFO - Chain [1] start processing


18:18:52 - cmdstanpy - INFO - Chain [1] done processing


18:18:52 - cmdstanpy - INFO - Chain [1] start processing


18:18:52 - cmdstanpy - INFO - Chain [1] done processing


18:18:52 - cmdstanpy - INFO - Chain [1] start processing


...

18:19:13 - cmdstanpy - INFO - Chain [1] start processing


18:19:13 - cmdstanpy - INFO - Chain [1] done processing


18:19:13 - cmdstanpy - INFO - Chain [1] start processing


18:19:14 - cmdstanpy - INFO - Chain [1] done processing


,modelo,MAE_USD,MAPE_percent
0,Prophet,6873.25,6.86
1,Baseline (dia anterior),1593.63,1.52


## 4. Modelo final e artefatos

Depois da avaliação, ajusto o modelo final com todo o histórico para que a API possa usá-lo. As métricas acima continuam sendo calculadas somente no período de teste.

In [5]:
final_model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
final_model.fit(series)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATH.write_text(model_to_json(final_model), encoding='utf-8')
metadata = {
    'source':data_source.get('source', 'fonte desconhecida'),
    'period_start':str(series.ds.min().date()), 'period_end':str(series.ds.max().date()),
    'total_rows':int(len(series)), 'train_rows':int(len(train)), 'test_rows':int(len(test)),
    'evaluation_method':'walk-forward de um dia à frente com janela expansiva',
    'prophet_mae_usd':round(float(metrics.iloc[0].MAE_USD), 2),
    'prophet_mape_percent':round(float(metrics.iloc[0].MAPE_percent), 2),
    'baseline_mae_usd':round(float(metrics.iloc[1].MAE_USD), 2),
    'baseline_mape_percent':round(float(metrics.iloc[1].MAPE_percent), 2), 'model':'Prophet'
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Modelo salvo:', ARTIFACT_PATH.resolve())
metadata

18:19:14 - cmdstanpy - INFO - Chain [1] start processing


18:19:14 - cmdstanpy - INFO - Chain [1] done processing


Modelo salvo: /workspace/artifacts/prophet_model.json


{'source': 'CryptoDataDownload / Binance BTCUSDT (fallback usado após limite de requisições do Yahoo Finance)',
 'period_start': '2024-01-01',
 'period_end': '2025-12-31',
 'total_rows': 731,
 'train_rows': 584,
 'test_rows': 147,
 'evaluation_method': 'walk-forward de um dia à frente com janela expansiva',
 'prophet_mae_usd': 6873.25,
 'prophet_mape_percent': 6.86,
 'baseline_mae_usd': 1593.63,
 'baseline_mape_percent': 1.52,
 'model': 'Prophet'}

## 5. Previsão do próximo dia

In [6]:
next_date = series.ds.max() + pd.Timedelta(days=1)
example = final_model.predict(pd.DataFrame({'ds':[next_date]}))
print('Data:', next_date.date())
print(f'Fechamento previsto (USD): {example.loc[0, "yhat"]:,.2f}')

Data: 2026-01-01
Fechamento previsto (USD): 80,521.30
